In [1]:
import os
import sys

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from utils import (
    build_binary_relevance_maps,
    build_eval_items,
    build_global_corpus,
    evaluate_sentence_transformer,
    load_triplets,
)


/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/sparql/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import os
import shutil
from pathlib import Path
from sentence_transformers import SentenceTransformer
from peft import PeftModel

# ---- required eval utils must already be imported in notebook ----
# from utils import load_triplets, build_global_corpus, build_eval_items, build_binary_relevance_maps, evaluate_sentence_transformer

BASE_MODEL = "Qwen/Qwen3-Embedding-0.6B"

DATA_PATH = os.path.join(PROJECT_ROOT, "dataset", "hard_triplets_tfidf_wikidata.json")
TOPKS = [3, 5, 10, 20]
BATCH = 32

# 1) Build eval objects ONCE
raw_data = load_triplets(DATA_PATH)
global_corpus, seen_q = build_global_corpus(raw_data)
eval_items = build_eval_items(raw_data, global_corpus, seen_q)
relevance_scores_per_item = build_binary_relevance_maps(eval_items, global_corpus)

print(f"Loaded {len(raw_data)} items")
print(f"Global corpus size: {len(global_corpus)}")
print(f"Eval anchors with non-empty positives: {len(eval_items)}")

# 2) Epoch checkpoints
CKPT_ROOT = Path(PROJECT_ROOT) / "../finetuning_llm/binary_classifier/checkpoints/qwen_wikidata_binary/checkpoints"
TMP_ROOT  = Path(PROJECT_ROOT) / "../finetuning_llm/binary_classifier/checkpoints/qwen_wikidata_binary/tmp_merged_eval"
TMP_ROOT.mkdir(parents=True, exist_ok=True)

steps_per_epoch = 12142
epochs = [1, 2, 3, 4, 5]

for epoch in epochs:
    step = epoch * steps_per_epoch
    adapter_dir = (CKPT_ROOT / f"checkpoint-{step}").resolve()
    merged_dir  = (TMP_ROOT / f"epoch_{epoch}_merged").resolve()

    if not adapter_dir.exists():
        print(f"\n[skip] epoch {epoch}: checkpoint not found -> {adapter_dir}")
        continue

    print(f"\n=== Epoch {epoch} | adapter: {adapter_dir} ===")

    # load base -> attach LoRA adapter -> merge
    model = SentenceTransformer(BASE_MODEL)
    word_module = model[0]
    peft_model = PeftModel.from_pretrained(word_module.auto_model, str(adapter_dir))
    word_module.auto_model = peft_model.merge_and_unload()

    # save merged temp model
    if merged_dir.exists():
        shutil.rmtree(merged_dir)
    model.save(str(merged_dir))

    # evaluate
    evaluate_sentence_transformer(
        model_name=str(merged_dir),
        display_name=f"Fine-tuned: Qwen wikidata (epoch {epoch})",
        eval_items=eval_items,
        global_corpus=global_corpus,
        relevance_scores_per_item=relevance_scores_per_item,
        topks=TOPKS,
        batch_size=BATCH,
    )

Loaded 186 items
Global corpus size: 2229
Eval anchors with non-empty positives: 186

=== Epoch 1 | adapter: /local/upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings/finetuning_llm/binary_classifier/checkpoints/qwen_wikidata_binary/checkpoints/checkpoint-12142 ===


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 5227.89it/s]



Fine-tuned: Qwen wikidata (epoch 1)
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.1649  0.0356  0.0554  0.1980  0.2796    0.1649  0.1582
@5     0.1742  0.0609  0.0841  0.2099  0.3280    0.1742  0.1672
@10    0.1403  0.1025  0.1100  0.2204  0.4032    0.1403  0.1530
@20    0.1075  0.1724  0.1222  0.2235  0.4462    0.1075  0.1579

=== Epoch 2 | adapter: /local/upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings/finetuning_llm/binary_classifier/checkpoints/qwen_wikidata_binary/checkpoints/checkpoint-24284 ===


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 7110.00it/s]



Fine-tuned: Qwen wikidata (epoch 2)
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.1183  0.0278  0.0426  0.1281  0.1774    0.1183  0.1126
@5     0.1290  0.0467  0.0644  0.1402  0.2312    0.1290  0.1222
@10    0.1129  0.0823  0.0886  0.1526  0.3226    0.1129  0.1199
@20    0.0957  0.1544  0.1101  0.1600  0.4194    0.0957  0.1324

=== Epoch 3 | adapter: /local/upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings/finetuning_llm/binary_classifier/checkpoints/qwen_wikidata_binary/checkpoints/checkpoint-36426 ===


...


Fine-tuned: Qwen wikidata (epoch 3)
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.1344  0.0312  0.0483  0.1649  0.2204    0.1344  0.1322
@5     0.1387  0.0524  0.0715  0.1786  0.2796    0.1387  0.1366
@10    0.1258  0.0963  0.1008  0.1968  0.4194    0.1258  0.1366
@20    0.1099  0.1705  0.1237  0.2023  0.4946    0.1099  0.1516

=== Epoch 4 | adapter: /local/upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings/finetuning_llm/binary_classifier/checkpoints/qwen_wikidata_binary/checkpoints/checkpoint-48568 ===


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 5569.86it/s]



Fine-tuned: Qwen wikidata (epoch 4)
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.1290  0.0284  0.0446  0.1694  0.2097    0.1290  0.1309
@5     0.1247  0.0453  0.0630  0.1806  0.2581    0.1247  0.1283
@10    0.1140  0.0870  0.0925  0.1941  0.3656    0.1140  0.1255
@20    0.1011  0.1648  0.1164  0.2026  0.4892    0.1011  0.1423

=== Epoch 5 | adapter: /local/upb/users/g/ganeshk/profiles/unix/cs/SPARQL-Embeddings/finetuning_llm/binary_classifier/checkpoints/qwen_wikidata_binary/checkpoints/checkpoint-60710 ===


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 5696.41it/s]



Fine-tuned: Qwen wikidata (epoch 5)
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.1254  0.0299  0.0460  0.1496  0.2097    0.1254  0.1208
@5     0.1194  0.0440  0.0606  0.1577  0.2473    0.1194  0.1185
@10    0.1097  0.0817  0.0870  0.1717  0.3495    0.1097  0.1189
@20    0.1022  0.1634  0.1169  0.1790  0.4516    0.1022  0.1390
